In [36]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from scipy.sparse import hstack
import re

In [37]:
import os
os.listdir("/content")


['.config', 'AI-Based Hiring Prediction System (1).csv', 'sample_data']

In [38]:
df = pd.read_csv("/content/AI-Based Hiring Prediction System (1).csv")
df.head()


,Resume_ID,Name,Skills,Experience (Years),Education,Certifications,Job Role,Recruiter Decision,Salary Expectation ($),Projects Count,AI Score (0-100)
0,1,Ashley Ali,"TensorFlow, NLP, Pytorch",10,B.Sc,NaN,AI Researcher,Hire,104895,8,100
1,2,Wesley Roman,"Deep Learning, Machine Learning, Python, SQL",10,MBA,Google ML,Data Scientist,Hire,113002,1,100
2,3,Corey Sanchez,"Ethical Hacking, Cybersecurity, Linux",1,MBA,Deep Learning Specialization,Cybersecurity Analyst,Hire,71766,7,70
3,4,Elizabeth Carney,"Python, Pytorch, TensorFlow",7,B.Tech,AWS Certified,AI Researcher,Hire,46848,0,95
4,5,Julie Hill,"SQL, React, Java",4,PhD,NaN,Software Engineer,Hire,87441,9,100


In [39]:
df.head()

,Resume_ID,Name,Skills,Experience (Years),Education,Certifications,Job Role,Recruiter Decision,Salary Expectation ($),Projects Count,AI Score (0-100)
0,1,Ashley Ali,"TensorFlow, NLP, Pytorch",10,B.Sc,NaN,AI Researcher,Hire,104895,8,100
1,2,Wesley Roman,"Deep Learning, Machine Learning, Python, SQL",10,MBA,Google ML,Data Scientist,Hire,113002,1,100
2,3,Corey Sanchez,"Ethical Hacking, Cybersecurity, Linux",1,MBA,Deep Learning Specialization,Cybersecurity Analyst,Hire,71766,7,70
3,4,Elizabeth Carney,"Python, Pytorch, TensorFlow",7,B.Tech,AWS Certified,AI Researcher,Hire,46848,0,95
4,5,Julie Hill,"SQL, React, Java",4,PhD,NaN,Software Engineer,Hire,87441,9,100


In [40]:
df.tail()

,Resume_ID,Name,Skills,Experience (Years),Education,Certifications,Job Role,Recruiter Decision,Salary Expectation ($),Projects Count,AI Score (0-100)
995,996,Brenda Williams,"Cybersecurity, Linux, Ethical Hacking",0,B.Sc,NaN,Cybersecurity Analyst,Reject,114364,9,60
996,997,Colleen Hicks,"Deep Learning, Machine Learning",0,MBA,Deep Learning Specialization,Data Scientist,Reject,103294,5,45
997,998,Michelle Molina,"TensorFlow, NLP",0,B.Tech,Google ML,AI Researcher,Hire,113855,9,65
998,999,Danielle Horn,"Linux, Networking, Cybersecurity, Ethical Hacking",8,PhD,AWS Certified,Cybersecurity Analyst,Hire,83146,10,100
999,1000,Chad Collins,"SQL, Machine Learning, Python, Deep Learning",7,M.Tech,Deep Learning Specialization,Data Scientist,Hire,119474,3,100


In [41]:
df.sample(5)

,Resume_ID,Name,Skills,Experience (Years),Education,Certifications,Job Role,Recruiter Decision,Salary Expectation ($),Projects Count,AI Score (0-100)
569,570,Gabriel Torres,"C++, SQL",2,M.Tech,AWS Certified,Software Engineer,Reject,83223,4,60
172,173,Brenda Crawford,"React, SQL, C++",3,B.Tech,Google ML,Software Engineer,Hire,100722,5,80
856,857,Karen Castillo,"SQL, Deep Learning, Python",5,MBA,Deep Learning Specialization,Data Scientist,Hire,46669,5,100
423,424,Cristian Hernandez,"C++, React, Java, SQL",7,B.Tech,AWS Certified,Software Engineer,Hire,59061,6,100
761,762,Shane Thomas,"Machine Learning, Deep Learning, SQL",6,M.Tech,NaN,Data Scientist,Hire,41610,7,100


In [42]:
df.shape

(1000, 11)

In [43]:
df.columns

Index(['Resume_ID', 'Name', 'Skills', 'Experience (Years)', 'Education',
       'Certifications', 'Job Role', 'Recruiter Decision',
       'Salary Expectation ($)', 'Projects Count', 'AI Score (0-100)'],
      dtype='object')

In [44]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 11 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   Resume_ID               1000 non-null   int64 
 1   Name                    1000 non-null   object
 2   Skills                  1000 non-null   object
 3   Experience (Years)      1000 non-null   int64 
 4   Education               1000 non-null   object
 5   Certifications          726 non-null    object
 6   Job Role                1000 non-null   object
 7   Recruiter Decision      1000 non-null   object
 8   Salary Expectation ($)  1000 non-null   int64 
 9   Projects Count          1000 non-null   int64 
 10  AI Score (0-100)        1000 non-null   int64 
dtypes: int64(5), object(6)
memory usage: 86.1+ KB


In [45]:
df["Recruiter Decision"].value_counts()

,count
Recruiter Decision,
Hire,812
Reject,188


In [46]:
df.describe()

,Resume_ID,Experience (Years),Salary Expectation ($),Projects Count,AI Score (0-100)
count,1000.000000,1000.000000,1000.000000,1000.00000,1000.000000
mean,500.500000,4.896000,79994.486000,5.13300,83.950000
std,288.819436,3.112695,23048.472549,3.23137,20.983036
min,1.000000,0.000000,40085.000000,0.00000,15.000000
25%,250.750000,2.000000,60415.750000,2.00000,70.000000
50%,500.500000,5.000000,79834.500000,5.00000,100.000000
75%,750.250000,8.000000,99583.250000,8.00000,100.000000
max,1000.000000,10.000000,119901.000000,10.00000,100.000000


In [47]:
df = df.drop(["Resume_ID", "Name"], axis=1)

In [48]:
df["Recruiter Decision"] = df["Recruiter Decision"].map({"Hire":1, "Reject":0})

In [49]:
df.isnull().sum()

,0
Skills,0
Experience (Years),0
Education,0
Certifications,274
Job Role,0
Recruiter Decision,0
Salary Expectation ($),0
Projects Count,0
AI Score (0-100),0


In [50]:
df.fillna("", inplace=True)

In [51]:
df["combined_text"] = df["Skills"] + " " + df["Certifications"] + " " + df["Job Role"]

In [52]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z ]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text

df["combined_text"] = df["combined_text"].apply(clean_text)

In [53]:
tfidf = TfidfVectorizer(max_features=3000)
text_features = tfidf.fit_transform(df["combined_text"])

In [54]:
le = LabelEncoder()
df["Education"] = le.fit_transform(df["Education"])

In [55]:
y = df["Recruiter Decision"]

X_numeric = df[[
    "Experience (Years)",
    "Salary Expectation ($)",
    "Projects Count",
    "Education"
]]

In [56]:
X_train_num, X_test_num, y_train, y_test = train_test_split(
    X_numeric, y, test_size=0.2, random_state=42
)

In [57]:
X_train_text, X_test_text = train_test_split(
    text_features, test_size=0.2, random_state=42
)


In [58]:
scaler = StandardScaler()
X_train_num = scaler.fit_transform(X_train_num)
X_test_num = scaler.transform(X_test_num)


In [59]:
X_train = hstack([X_train_text, X_train_num])
X_test = hstack([X_test_text, X_test_num])


In [60]:
models = {
    "Logistic Regression": LogisticRegression(),
    "Random Forest": RandomForestClassifier(),
    "SVM": SVC(probability=True),
    "KNN": KNeighborsClassifier()
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    results[name] = acc

    print(name)
    print("Accuracy:", acc)
    print(classification_report(y_test, preds))


Logistic Regression
Accuracy: 0.975
              precision    recall  f1-score   support

           0       0.94      0.96      0.95        46
           1       0.99      0.98      0.98       154

    accuracy                           0.97       200
   macro avg       0.96      0.97      0.96       200
weighted avg       0.98      0.97      0.98       200

Random Forest
Accuracy: 0.95
              precision    recall  f1-score   support

           0       1.00      0.78      0.88        46
           1       0.94      1.00      0.97       154

    accuracy                           0.95       200
   macro avg       0.97      0.89      0.92       200
weighted avg       0.95      0.95      0.95       200

SVM
Accuracy: 0.965
              precision    recall  f1-score   support

           0       0.93      0.91      0.92        46
           1       0.97      0.98      0.98       154

    accuracy                           0.96       200
   macro avg       0.95      0.95      0.95

In [61]:
pd.DataFrame(results.items(), columns=["Model", "Accuracy"])


,Model,Accuracy
0,Logistic Regression,0.975
1,Random Forest,0.950
2,SVM,0.965
3,KNN,0.935


In [62]:
pipe = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", LogisticRegression())
])

params = {"clf__C": [0.1, 1, 10]}

grid = GridSearchCV(pipe, params, cv=5)
grid.fit(df["combined_text"], y)

grid.best_params_


{'clf__C': 0.1}

In [63]:
def predict_hiring(skills, experience, education, certs, projects, salary):

    text = clean_text(skills + " " + certs)
    text_vec = tfidf.transform([text])

    edu_encoded = le.transform([education])[0]

    num_data = scaler.transform([[experience, salary, projects, edu_encoded]])

    final_input = hstack([text_vec, num_data])

    model = models["Random Forest"]
    prob = model.predict_proba(final_input)[0][1]
    pred = model.predict(final_input)[0]

    return ("Hire" if pred==1 else "Reject", prob)


In [64]:
predict_hiring(
    skills="Python SQL Machine Learning",
    experience=3,
    education="M.Tech",
    certs="AWS Data Science",
    projects=5,
    salary=60000
)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


('Hire', np.float64(0.85))

In [65]:
le.classes_


array(['B.Sc', 'B.Tech', 'M.Tech', 'MBA', 'PhD'], dtype=object)